In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
_r = pathlib.Path.cwd().resolve()
while not (_r / "config" / "settings.yaml").exists() and _r != _r.parent:
    _r = _r.parent
sys.path.insert(0, str(_r))                      # makes `import src...` work from any folder
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from src.notebook_utils import *
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 40, "display.width", 140)
print("project root:", _r)

project root: /app


# Statistical analysis
Formal tests supporting the research questions. Significance level α = 0.05 with Bonferroni correction across the family of tests.

In [2]:
from src.model_evaluation import statistics as st
from src.model_evaluation.evaluate import bootstrap_ci
df = load_clean(); fe = load_features(); summ = load_summary()

In [3]:
# H1: fraud amounts differ from legitimate amounts (non-parametric)
h1 = st.mann_whitney(df[df.is_fraud==1].amount, df[df.is_fraud==0].amount); h1

{'U': 42295035.5,
 'p_value': 1.5591932744106755e-32,
 'rank_biserial': -0.1994839033053717,
 'n_a': 1206,
 'n_b': 58476}

In [4]:
# H2: channel and fraud are associated
h2 = st.chi_square(df, 'channel', 'is_fraud'); h2

{'chi2': 159.1122048002272,
 'dof': 3,
 'p_value': 2.849179051497302e-34,
 'cramers_v': 0.0516333214055073}

In [5]:
# H3: authentication method and fraud are associated
h3 = st.chi_square(df, 'auth_method', 'is_fraud'); h3

{'chi2': 134.32519345647702,
 'dof': 3,
 'p_value': 6.322246922387462e-29,
 'cramers_v': 0.04744135156186112}

In [6]:
sv = load_survey(); sc = load_trust(); full = sv.merge(sc[['respondent_id','trust_score']], on='respondent_id')
h4 = st.kruskal(full, 'trust_score', 'age_band'); h5 = st.mann_whitney(full[full.experienced_fraud==1].trust_score, full[full.experienced_fraud==0].trust_score)
h4, h5

({'H': 2.256492615445407, 'p_value': 0.688701046549596, 'groups': 5},
 {'U': 6177.5,
  'p_value': 2.4787678078370646e-05,
  'rank_biserial': 0.336608676975945,
  'n_a': 64,
  'n_b': 291})

In [7]:
tests = {'H1 amount': h1['p_value'], 'H2 channel': h2['p_value'], 'H3 auth': h3['p_value'], 'H4 age trust': h4['p_value'], 'H5 fraud-experience trust': h5['p_value']}
m = len(tests)
pd.DataFrame({'p': tests, 'bonferroni_p': {k: min(v*m, 1) for k, v in tests.items()}, 'significant@0.05': {k: v*m < .05 for k, v in tests.items()}})

,p,bonferroni_p,significant@0.05
H1 amount,1.559193e-32,7.795966e-32,True
H2 channel,2.849179e-34,1.424590e-33,True
H3 auth,6.322247e-29,3.161123e-28,True
H4 age trust,6.887010e-01,1.000000e+00,False
H5 fraud-experience trust,2.478768e-05,1.239384e-04,True


In [8]:
# Model comparison: McNemar (paired errors) and bootstrap CI for the champion
print('McNemar champion vs LR:', summ['models']['mcnemar_vs_lr'])
print('Champion ROC-AUC 95% CI:', summ['models']['champion_roc_auc_ci95'])

McNemar champion vs LR: {'b': 30, 'c': 8, 'chi2': 11.605263157894736, 'p_value': 0.0006576543953178447}
Champion ROC-AUC 95% CI: [0.8471476461231401, 0.9181854459621215]


Effect sizes (Cramér's V, rank-biserial) are reported alongside p-values: with tens of thousands of rows, tiny effects become 'significant', so magnitude matters.